# PyHamilton Demo: Fill a 96-Well Plate with Water

This notebook will:

1. Pick up 8 tips
2. Aspirate 700 uL of water from the **center trough**
3. Dispense into each well of the 96-well plate, one column at a time

Turn on VENUS **Simulation** mode for a dry run.

## 1. Imports

In [37]:
from pyhamilton import (
    HamiltonInterface,
    INITIALIZE,
    LayoutManager,
    layout_item,
    Tip96,
    Plate96,
    Reservoir60mL,
    initialize,
    tip_pick_up,
    aspirate,
    dispense,
    tip_eject,
)

## 2. Load the deck layout

The layout (`.lay`) file was made in VENUS. It tells pyhamilton what labware is on the deck and where it sits.

In [38]:
layout_file = r"C:\Program Files (x86)\HAMILTON\Methods\Demo\msacl-2026-demo.lay"

lmgr = LayoutManager(layout_file)

## 3. Find our labware

Each piece of labware has a name in the layout file. You can see these names in the VENUS deck layout editor.

In [39]:
# 1000 uL tips (first tip rack)
tips = layout_item(lmgr, Tip96, 'htf_l_0001')

# Center trough (position 2 of the 3-trough carrier).
# pyhamilton has no 120 mL trough class. Reservoir60mL works fine here:
# it gives us 8 spots in one column, one for each channel.
trough = layout_item(lmgr, Reservoir60mL, 'rgt_cont_120ml_BC_A00_0002')

# 2 mL deep-well 96-well plate
plate = layout_item(lmgr, Plate96, 'Cos_96_DW_2mL_0001')

print('Tips:  ', tips.layout_name())
print('Trough:', trough.layout_name())
print('Plate: ', plate.layout_name())

Tips:   htf_l_0001
Trough: rgt_cont_120ml_BC_A00_0002
Plate:  Cos_96_DW_2mL_0001


## 4. Settings

In [40]:
volume = 700  # uL per well

# A liquid class tells the robot how to pipette (speeds, air gaps, etc.)
liquid_class = 'HighVolumeFilter_Water_DispenseJet_Empty'

## 5. Positions

A position is `(labware, index)`. The robot has 8 channels, so we use lists of 8 positions.

Indexes start at 0 and run **down each column**:

| index | 0 | 1 | ... | 7 | 8 | 9 | ... | 95 |
|---|---|---|---|---|---|---|---|---|
| well | A1 | B1 | ... | H1 | A2 | B2 | ... | H12 |

In [41]:
# Column 1 of the tip rack
tip_positions = [
    (tips, 0),
    (tips, 1),
    (tips, 2),
    (tips, 3),
    (tips, 4),
    (tips, 5),
    (tips, 6),
    (tips, 7),
]

# 8 spots in the trough, one for each channel
trough_positions = [
    (trough, 0),
    (trough, 1),
    (trough, 2),
    (trough, 3),
    (trough, 4),
    (trough, 5),
    (trough, 6),
    (trough, 7),
]

# One volume for each channel
volumes = [volume, volume, volume, volume, volume, volume, volume, volume]

## 6. Connect to the robot

This opens the VENUS Run Control window. `initialize` gets the robot ready to run.

In [42]:
ham_int = HamiltonInterface(windowed=True)
ham_int.start()
ham_int.wait_on_response(ham_int.send_command(INITIALIZE))

#initialize(ham_int)

Starting a new server thread
Sending ping to check if interface is open
Opening HSL application
started the server thread


HamiltonResponse(status=<HamiltonResponseStatus.SUCCESS: 2>, return_data=[], moduleID='', parsed_return=[{'numField': 1, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste', 'labwarePos': ' '}, {'numField': 2, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste', 'labwarePos': ' '}, {'numField': 3, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste', 'labwarePos': ' '}, {'numField': 4, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste', 'labwarePos': ' '}, {'numField': 5, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste', 'labwarePos': ' '}, {'numField': 6, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste', 'labwarePos': ' '}, {'numField': 7, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': ' ', 'labwareName': 'Waste'

## 7. Pick up tips

In [43]:
tip_pick_up(ham_int, tip_positions)

## 8. Fill column 1 (wells A1 to H1)

In [44]:
plate_column_1 = [
    (plate, 0),  # A1
    (plate, 1),  # B1
    (plate, 2),  # C1
    (plate, 3),  # D1
    (plate, 4),  # E1
    (plate, 5),  # F1
    (plate, 6),  # G1
    (plate, 7),  # H1
]

aspirate(ham_int, trough_positions, volumes, liquidClass=liquid_class)
dispense(ham_int, plate_column_1, volumes, liquidClass=liquid_class)

DispenseResult(liquidHeights=[111.1, 111.2, 111.3, 111.4, 111.5, 111.6, 111.7, 111.8], liquidVolumes=[-1.911345, -1.908832, -1.906318, -1.903805, -1.901292, -1.898779, -1.896265, -1.893752], raw=HamiltonResponse(status=<HamiltonResponseStatus.SUCCESS: 2>, return_data=['111.1000;111.2000;111.3000;111.4000;111.5000;111.6000;111.7000;111.8000', '-1.911345;-1.908832;-1.906318;-1.903805;-1.901292;-1.898779;-1.896265;-1.893752'], moduleID='111.1000;111.2000;111.3000;111.4000;111.5000;111.6000;111.7000;111.8000', parsed_return=[{'numField': 1, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': '700.0', 'labwareName': 'Cos_96_DW_2mL_0001', 'labwarePos': 'A1'}, {'numField': 2, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': '700.0', 'labwareName': 'Cos_96_DW_2mL_0001', 'labwarePos': 'B1'}, {'numField': 3, 'mainErrField': 0, 'slaveErr': 0, 'recoveryBtnId': 0, 'stepData': '700.0', 'labwareName': 'Cos_96_DW_2mL_0001', 'labwarePos': 'C1'}, {'numField': 4, 'mainErrFi

## 9. Fill columns 2 to 12

These are the same steps as column 1. The loop just changes which wells we use.

We keep the same tips for every column because it's only water.

In [45]:
for column in range(1, 12):  # 1 = column 2, ..., 11 = column 12
    first_well = column * 8

    plate_column = [
        (plate, first_well + 0),
        (plate, first_well + 1),
        (plate, first_well + 2),
        (plate, first_well + 3),
        (plate, first_well + 4),
        (plate, first_well + 5),
        (plate, first_well + 6),
        (plate, first_well + 7),
    ]

    print('Filling column', column + 1)
    aspirate(ham_int, trough_positions, volumes, liquidClass=liquid_class)
    dispense(ham_int, plate_column, volumes, liquidClass=liquid_class)

Filling column 2
Filling column 3
Filling column 4
Filling column 5
Filling column 6
Filling column 7
Filling column 8
Filling column 9
Filling column 10
Filling column 11
Filling column 12


## 10. Put the tips back

The tips go back into the same rack spots so the demo can run again.

In [46]:
tip_eject(ham_int, tip_positions)

## 11. Disconnect

In [47]:
ham_int.stop()

Sending end command
Stopping server thread
Force closing HTTP server
Server thread run() method completed
Joining server thread with timeout
Server thread exited successfully
